In [1]:
import pandas as pd

DATA_DIR = '/kaggle/input/datasets/abdullahehdayb/olist-abdullah-practice/'

customers = pd.read_csv(DATA_DIR + 'customers.csv')
orders = pd.read_csv(DATA_DIR + 'orders.csv')
order_items = pd.read_csv(DATA_DIR + 'order_items.csv')
order_payments = pd.read_csv(DATA_DIR + 'order_payments.csv')
order_reviews = pd.read_csv(DATA_DIR + 'order_reviews.csv')
products = pd.read_csv(DATA_DIR + 'products.csv')
sellers = pd.read_csv(DATA_DIR + 'sellers.csv')
geolocation = pd.read_csv(DATA_DIR + 'geolocation.csv')
category_translation = pd.read_csv(DATA_DIR + 'product_category_name_translation.csv')

tables = {
    'customers': customers, 'orders': orders, 'order_items': order_items,
    'order_payments': order_payments, 'order_reviews': order_reviews,
    'products': products, 'sellers': sellers, 'geolocation': geolocation,
    'category_translation': category_translation
}

for name, df in tables.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

customers: 99441 rows, 5 columns
orders: 99441 rows, 8 columns
order_items: 112650 rows, 7 columns
order_payments: 103886 rows, 5 columns
order_reviews: 99224 rows, 7 columns
products: 32951 rows, 9 columns
sellers: 3095 rows, 4 columns
geolocation: 1000163 rows, 5 columns
category_translation: 71 rows, 2 columns


In [2]:
print("customers - unique customer_id:", customers['customer_id'].nunique(), "vs rows:", len(customers))
print("orders - unique order_id:", orders['order_id'].nunique(), "vs rows:", len(orders))
print("order_items - duplicate rows:", order_items.duplicated().sum())
print("order_items - unique order_id:", order_items['order_id'].nunique(), "(should be < total rows)")
print("order_payments - unique order_id:", order_payments['order_id'].nunique(), "(should be <= total rows)")
print("order_reviews - unique order_id:", order_reviews['order_id'].nunique(), "vs rows:", len(order_reviews))
print("products - unique product_id:", products['product_id'].nunique(), "vs rows:", len(products))
print("sellers - unique seller_id:", sellers['seller_id'].nunique(), "vs rows:", len(sellers))

customers - unique customer_id: 99441 vs rows: 99441
orders - unique order_id: 99441 vs rows: 99441
order_items - duplicate rows: 0
order_items - unique order_id: 98666 (should be < total rows)
order_payments - unique order_id: 99440 (should be <= total rows)
order_reviews - unique order_id: 98673 vs rows: 99224
products - unique product_id: 32951 vs rows: 32951
sellers - unique seller_id: 3095 vs rows: 3095


In [3]:
print("order_reviews - duplicate rows:", order_reviews.duplicated().sum())
print("order_reviews - duplicate review_id:", order_reviews['review_id'].duplicated().sum())

dupe_orders = order_reviews['order_id'].value_counts()
dupe_orders = dupe_orders[dupe_orders > 1]
print(f"\nOrders with multiple reviews: {len(dupe_orders)}")
order_reviews[order_reviews['order_id'] == dupe_orders.index[0]]

order_reviews - duplicate rows: 0
order_reviews - duplicate review_id: 814

Orders with multiple reviews: 547


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
8274,b04ed893318da5b863e878cd3d0511df,03c939fd7fd3b38f8485a0f95798f1f6,3,NaN,Um ponto negativo que achei foi a cobrança de ...,2018-03-20 00:00:00,2018-03-21 02:28:23
51536,f4bb9d6dd4fb6dcc2298f0e7b17b8e1e,03c939fd7fd3b38f8485a0f95798f1f6,4,NaN,NaN,2018-03-29 00:00:00,2018-03-30 00:29:09
69449,405eb2ea45e1dbe2662541ae5b47e2aa,03c939fd7fd3b38f8485a0f95798f1f6,3,NaN,Seria ótimo se tivesem entregue os 3 (três) pe...,2018-03-06 00:00:00,2018-03-06 19:50:32


In [4]:
items_agg = order_items.groupby('order_id').agg(
    n_items=('order_item_id', 'count'),
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    n_distinct_products=('product_id', 'nunique'),
    n_distinct_sellers=('seller_id', 'nunique')
).reset_index()

payments_agg = order_payments.groupby('order_id').agg(
    total_payment=('payment_value', 'sum'),
    n_payment_entries=('payment_value', 'count'),
    max_installments=('payment_installments', 'max')
).reset_index()

ml_table = orders.merge(customers, on='customer_id', how='left')
ml_table = ml_table.merge(items_agg, on='order_id', how='left')
ml_table = ml_table.merge(payments_agg, on='order_id', how='left')

print("Final shape:", ml_table.shape)
print("Unique order_id:", ml_table['order_id'].nunique())

Final shape: (99441, 20)
Unique order_id: 99441


In [5]:
print(ml_table.head())
print(ml_table.isnull().sum())

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08

In [6]:
ml_table[ml_table['n_items'].isnull()]['order_status'].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [7]:
ml_table.to_csv('/kaggle/working/ml_table_notebook1.csv', index=False)
print("Saved:", ml_table.shape)

Saved: (99441, 20)
